# Municipal photo pilot, private draft
No upload or training has been run. Targets: visible violation labels + municipality classification when verified, vehicle plate and offending-vehicle box; optional recorded make/model/colour. No enforcement outcomes.

Use a PRIVATE dataset AND PRIVATE notebook after owner approval. Images may show plates, faces and recognizable locations; metadata stripping is not anonymization. Never attach local-private/source-index.json or the original repository.

This notebook is a guarded baseline, not a trained model or accuracy claim. It refuses unreviewed labels/boxes and missing location-disjoint splits. Plate OCR uses a separate existing reader first; no fabricated OCR targets. Optional vehicle metadata remains candidate-only until reviewed.

In [ ]:
from pathlib import Path
import json, hashlib, random
ROOT = Path('/kaggle/input/REPLACE_WITH_APPROVED_PRIVATE_DATASET')
manifest = json.loads((ROOT / 'manifest.reviewed.json').read_text())
LABELS = ['sidewalk_parking', 'red_white', 'crosswalk_obstruction']
assert manifest, 'No reviewed manifest'
for row in manifest:
    assert row['review_status'] == 'approved_visual_event_and_target', 'Review images/events/target boxes first'
    assert row['split'] in {'train','val','test'}
    assert row['target_vehicle_box'] is not None
    assert all(row['label_state'][label] in {'present','absent'} for label in LABELS)
    path = ROOT / row['image']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row['image_sha256']
# Whole-location/event groups never cross partitions.
seen={}
for row in manifest:
    assert seen.setdefault(row['group_id'], row['split']) == row['split'], 'Group leakage'
assert {r['split'] for r in manifest} == {'train','val','test'}


## Baseline visual classifier
Pretrained ResNet18, frozen feature extractor, three sigmoid outputs. Only human-reviewed visible labels become targets. Use weighted BCE for imbalance. City classification is a separate routing target, not an automatic replacement for visual labels. A label present in an acknowledgment is usable only with verified record mapping and reviewed catalog matching. Flag filed-code disagreement rather than rewriting history.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
random.seed(329); torch.manual_seed(329)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
transform = models.ResNet18_Weights.DEFAULT.transforms()
class Photos(Dataset):
    def __init__(self, split): self.rows=[r for r in manifest if r['split']==split]
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        with Image.open(ROOT/r['image']) as image: x=transform(image.convert('RGB'))
        y=torch.tensor([r['label_state'][k]=='present' for k in LABELS],dtype=torch.float32)
        return x,y
train=Photos('train'); val=Photos('val'); test=Photos('test')
assert min(len(train),len(val),len(test))>0
# Weight download occurs only during separately approved notebook execution.
model=models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters(): p.requires_grad=False
model.fc=nn.Linear(model.fc.in_features,len(LABELS));model.to(device)
y=torch.stack([train[i][1] for i in range(len(train))]); pos=y.sum(0)
assert torch.all(pos>0) and torch.all(pos<len(train)), 'Need positive and negative examples per class'
loss_fn=nn.BCEWithLogitsLoss(pos_weight=((len(train)-pos)/pos).to(device))
optimizer=torch.optim.Adam(model.fc.parameters(),lr=1e-3)
train_loader=DataLoader(train,batch_size=16,shuffle=True,num_workers=0)
for epoch in range(5):
    model.train();model.bn1.eval() # Frozen backbone statistics should not update.
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d): module.eval()
    for x,y in train_loader:
        optimizer.zero_grad();loss=loss_fn(model(x.to(device)),y.to(device));loss.backward();optimizer.step()
print('Baseline trained. Evaluate validation thresholds before opening test results.')


In [ ]:
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
import numpy as np
@torch.no_grad()
def predict(ds):
    model.eval();probs=[];truth=[]
    for x,y in DataLoader(ds,batch_size=16,num_workers=0):
        probs.append(model(x.to(device)).sigmoid().cpu().numpy());truth.append(y.numpy())
    return np.concatenate(probs),np.concatenate(truth)
vp,vy=predict(val)
thresholds=[]
for i in range(len(LABELS)):
    scores=[]
    for threshold in np.arange(.1,.91,.05):
        _,_,f,_=precision_recall_fscore_support(vy[:,i],vp[:,i]>=threshold,average='binary',zero_division=0)
        scores.append((f,threshold))
    thresholds.append(max(scores)[1])
tp,ty=predict(test)
p,r,f,s=precision_recall_fscore_support(ty,tp>=np.array(thresholds),average=None,zero_division=0)
for i,label in enumerate(LABELS):
    print(label,dict(precision=float(p[i]),recall=float(r[i]),f1=float(f[i]),support=int(s[i]),threshold=float(thresholds[i])))
# Report group counts and exclusions, not only image counts. No legal-violation accuracy claim.


## Vehicle selection, plate OCR and optional type
1. Review target boxes in normalized [x1,y1,x2,y2], one offending vehicle or an explicit multi-target list. Do not equate foreground vehicle with reported vehicle.
2. Feed reviewed plate crops to an existing licensed OCR reader. Record exact visible characters, preserving leading zeros. Unknown/occluded digits stay unknown, never filled from registry.
3. Compare exact normalized plate match and character error rate on whole-location holdout. Report box IoU and target-selection errors separately.
4. Optional make/model/colour target uses only reviewed visible/recorded evidence, not an unsupported inference. No model training code for these heads is claimed complete here.
5. City-classification head requires enough verified labeled examples and reviewed code/name reconciliation; no enforcement label or automatic filing is included.

Release gate: owner content/privacy approval, image inspection, target/plate annotation, independent code review, local container tests, restored CI billing and legal/data-use review as needed. No upload tool or API key is part of this notebook.